In [16]:
import polars as pl

In [17]:
df = pl.read_csv('2_residenceAnalysis/data/durations.csv')

In [18]:
size = df.height

In [19]:
df = df.with_columns(
    pl.int_range(size).add(1).alias('duration'),
)
df

1.00_10,1.00_20,1.00_40,1.00_60,2.80_10,2.80_20,2.80_40,2.80_60,3.00_10,3.00_20,3.00_40,3.00_60,3.50_10,3.50_20,3.50_40,3.50_60,4.00_10,4.00_20,4.00_40,4.00_60,duration
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
142333,261598,506462,827412,158876,255617,387030,528478,133913,206111,268032,302901,53716,31774,29436,30790,2821,3106,2340,2464,1
92429,164720,286792,411299,110901,178297,268098,365155,99649,154194,200927,229424,44969,26884,25185,26646,2610,2672,2176,2257,2
72216,128329,219653,307051,82092,131264,194687,263706,76114,118622,156803,179130,38690,22761,21927,23355,2309,2486,2021,2086,3
58183,102797,176594,243013,62063,99084,147051,199968,59713,93352,124746,144388,33025,19949,19149,20766,2191,2231,1851,1964,4
46558,83068,142351,195746,48120,77052,115167,156237,47378,74593,100769,118628,28606,17366,16834,18623,2015,2135,1803,1769,5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23996
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23997
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23998


In [20]:
frame = df.select(
    pl.col('duration'),
    pl.all().exclude('duration'),
)
frame

duration,1.00_10,1.00_20,1.00_40,1.00_60,2.80_10,2.80_20,2.80_40,2.80_60,3.00_10,3.00_20,3.00_40,3.00_60,3.50_10,3.50_20,3.50_40,3.50_60,4.00_10,4.00_20,4.00_40,4.00_60
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,142333,261598,506462,827412,158876,255617,387030,528478,133913,206111,268032,302901,53716,31774,29436,30790,2821,3106,2340,2464
2,92429,164720,286792,411299,110901,178297,268098,365155,99649,154194,200927,229424,44969,26884,25185,26646,2610,2672,2176,2257
3,72216,128329,219653,307051,82092,131264,194687,263706,76114,118622,156803,179130,38690,22761,21927,23355,2309,2486,2021,2086
4,58183,102797,176594,243013,62063,99084,147051,199968,59713,93352,124746,144388,33025,19949,19149,20766,2191,2231,1851,1964
5,46558,83068,142351,195746,48120,77052,115167,156237,47378,74593,100769,118628,28606,17366,16834,18623,2015,2135,1803,1769
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
23996,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
23997,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
23998,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [21]:
frame.write_csv('2_residenceAnalysis/data/durationVScounts.csv')

In [22]:
mframe = frame.unpivot(index='duration', variable_name='case', value_name='count')

In [23]:
mframe.head()

duration,case,count
i64,str,i64
1,"""1.00_10""",142333
2,"""1.00_10""",92429
3,"""1.00_10""",72216
4,"""1.00_10""",58183
5,"""1.00_10""",46558


In [24]:
mframe = mframe.with_columns(
    pl.col('case').str.extract(r'^([^_]+)_([^_]+)$', 1).cast(pl.Float32).alias('energy(kT)'),
    pl.col('case').str.extract(r'^([^_]+)_([^_]+)$', 2).cast(pl.Float32).alias('concentration(uM)'),
)
mframe = mframe.drop('case')
mframe

duration,count,energy(kT),concentration(uM)
i64,i64,f32,f32
1,142333,1.0,10.0
2,92429,1.0,10.0
3,72216,1.0,10.0
4,58183,1.0,10.0
5,46558,1.0,10.0
…,…,…,…
23996,0,4.0,60.0
23997,0,4.0,60.0
23998,0,4.0,60.0


In [27]:
mframe = mframe.filter(
    (pl.col('count') > 0.0)
)
mframe

duration,count,energy(kT),concentration(uM)
i64,i64,f32,f32
1,142333,1.0,10.0
2,92429,1.0,10.0
3,72216,1.0,10.0
4,58183,1.0,10.0
5,46558,1.0,10.0
…,…,…,…
7923,1,4.0,60.0
8289,1,4.0,60.0
8357,1,4.0,60.0


In [26]:
mframe.write_csv('2_residenceAnalysis/data/durationVScounts_melted.csv')